In [27]:
from pathlib import Path
import os

import polars as pl

import pandas as pd
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font
from copy import copy


os.chdir('/home/vladimirnoz/Projects/Codebook_Perspectives/AS_CHS_GHTS/')

TF_TABLES_PATH = Path('as_tables/')
JOINT_PATH = Path('joint_tables/ASB_Phenotypes/selex+chipseq.tsv')
OUTPUT_DIR_PATH = Path('collection/')
FDR_THR = 0.05


fdr_filter = (pl.col('fdr_comb_pval') < 0.05)

In [28]:
tables_list = []

for path in TF_TABLES_PATH.glob('*/ASB_motifs/*.tsv'):
    tf, exp = path.stem, path.parent.parent.stem
    local_df = pl.read_csv(path, separator='\t', infer_schema_length=10000)
    local_df = local_df.filter(fdr_filter).with_columns(
        pl.lit(tf).alias('tf'),
        pl.lit(exp).alias('experiment_type'),
    )
    tables_list.append(local_df)

tf_table = pl.concat(tables_list, how='vertical_relaxed')
tf_table.schema

Schema([('#chr', String),
        ('start', Int64),
        ('end', Int64),
        ('mean_bad', Float64),
        ('id', String),
        ('max_cover', Int64),
        ('ref', String),
        ('alt', String),
        ('n_reps', Int64),
        ('bads', String),
        ('scorefiles', String),
        ('ref_counts', String),
        ('alt_counts', String),
        ('ref_es', String),
        ('alt_es', String),
        ('ref_pval', String),
        ('alt_pval', String),
        ('ref_comb_es', Float64),
        ('alt_comb_es', Float64),
        ('ref_comb_pval', Float64),
        ('alt_comb_pval', Float64),
        ('ref_fdr_comb_pval', Float64),
        ('alt_fdr_comb_pval', Float64),
        ('pref_allele', String),
        ('comb_es', Float64),
        ('comb_pval', Float64),
        ('fdr_comb_pval', Float64),
        ('ref_motif_pos', Int64),
        ('ref_motif_orient', String),
        ('alt_motif_pos', Int64),
        ('alt_motif_orient', String),
        ('ref_motif_pval', Fl

In [29]:
tf_table = tf_table.with_columns(
    (
        pl.when(pl.col('motif_conc') == 'No Hit')
            .then(pl.lit('No Hit'))
            .when(pl.col('motif_fc').abs() < 1)
            .then(pl.lit('Weak ') + pl.col('motif_conc'))
            .otherwise(pl.col('motif_conc'))
    ).alias('motif_conc')
)

In [34]:
tf_table.group_by(
    ['experiment_type', 'motif_conc',]
).len().sort(by=['experiment_type', 'motif_conc'])

experiment_type,motif_conc,len
str,str,u32
"""chipseq""","""Concordant""",1373
"""chipseq""","""Discordant""",436
"""chipseq""","""No Hit""",7766
"""chipseq""","""Weak Concordant""",524
"""chipseq""","""Weak Discordant""",476
"""selex""","""Concordant""",309
"""selex""","""Discordant""",142
"""selex""","""No Hit""",730
"""selex""","""Weak Concordant""",160


In [30]:
table = pl.read_csv(JOINT_PATH, separator='\t', infer_schema_length=10000).filter(fdr_filter)
table = table.drop(pl.col('ADASTRA_CL'))
print(len(table))
table.schema

22064


Schema([('#chr', String),
        ('start', Int64),
        ('end', Int64),
        ('mean_bad', Float64),
        ('id', String),
        ('max_cover', Int64),
        ('ref', String),
        ('alt', String),
        ('n_reps', Int64),
        ('ref_comb_es', Float64),
        ('alt_comb_es', Float64),
        ('ref_comb_pval', Float64),
        ('alt_comb_pval', Float64),
        ('ref_fdr_comb_pval', Float64),
        ('alt_fdr_comb_pval', Float64),
        ('pref_allele', String),
        ('comb_es', Float64),
        ('comb_pval', Float64),
        ('fdr_comb_pval', Float64),
        ('ebi', String),
        ('eQTL_cis', String),
        ('eQTL_cis_gene', String),
        ('ADASTRA_TF', String)])

In [31]:
tf_table_pd = tf_table.to_pandas()
table_pd = table.to_pandas()

desc_wb = load_workbook(OUTPUT_DIR_PATH / 'descriptions.xlsx')

wb = Workbook()
wb.remove(wb.active)

desc1_source = desc_wb['Description 1']
ws1 = wb.create_sheet('TF-wise data description', 0)
for row in desc1_source.iter_rows():
    for cell in row:
        new_cell = ws1[cell.coordinate]
        new_cell.value = cell.value
        if cell.has_style:
            new_cell.font = copy(cell.font)
            new_cell.alignment = copy(cell.alignment)

for row_num, row_dim in desc1_source.row_dimensions.items():
    ws1.row_dimensions[row_num].height = row_dim.height
for col_letter, col_dim in desc1_source.column_dimensions.items():
    ws1.column_dimensions[col_letter].width = col_dim.width

for merged_range in desc1_source.merged_cells.ranges:
    ws1.merge_cells(str(merged_range))

ws2 = wb.create_sheet('TF-wise data', 1)
for c_idx, col_name in enumerate(tf_table_pd.columns, 1):
    cell = ws2.cell(row=1, column=c_idx, value=col_name)
    cell.font = Font(bold=True)
for r_idx, row in enumerate(tf_table_pd.values, 2):
    for c_idx, value in enumerate(row, 1):
        ws2.cell(row=r_idx, column=c_idx, value=value)

desc2_source = desc_wb['Description 2']
ws3 = wb.create_sheet('Joint data description', 2)
for row in desc2_source.iter_rows():
    for cell in row:
        new_cell = ws3[cell.coordinate]
        new_cell.value = cell.value
        if cell.has_style:
            new_cell.font = copy(cell.font)
            new_cell.alignment = copy(cell.alignment)

for row_num, row_dim in desc2_source.row_dimensions.items():
    ws3.row_dimensions[row_num].height = row_dim.height
for col_letter, col_dim in desc2_source.column_dimensions.items():
    ws3.column_dimensions[col_letter].width = col_dim.width

for merged_range in desc2_source.merged_cells.ranges:
    ws3.merge_cells(str(merged_range))

ws4 = wb.create_sheet('Joint data', 3)
for c_idx, col_name in enumerate(table_pd.columns, 1):
    cell = ws4.cell(row=1, column=c_idx, value=col_name)
    cell.font = Font(bold=True)
for r_idx, row in enumerate(table_pd.values, 2):
    for c_idx, value in enumerate(row, 1):
        ws4.cell(row=r_idx, column=c_idx, value=value)

wb.save(OUTPUT_DIR_PATH / 'collection.xlsx')
print("Complete Excel file created successfully!")
print("Sheets: 1) TF-wise data description, 2) TF-wise data, 3) Joint data description, 4) Joint data")

Complete Excel file created successfully!
Sheets: 1) TF-wise data description, 2) TF-wise data, 3) Joint data description, 4) Joint data
